In [1]:
"""
exp07 Step 01: Warning Lights Exploration (EXPLORATION ON 2005-2014 ONLY; context, PROTOCOL.md §9)

The 3 candidates of PROTOCOL.md §5 (sell when at least k of 5 warning lights are on at the 15:58 decision, buy back when
fewer than k - 1 are on; k in {3, 4, 5}) are replayed as ONE continuous path over 2005-01-03 -> 2014-12-31 (start
invested, no forced buy-back), against buy-and-hold over the same sessions. The data is cut at
so.config.EXPLORATION_DATA_CUTOFF_DATE_STR (2015-03-18): nothing of the walk-forward period is read.

What it shows: how often the lights agree, and every exit episode (expected 3-8 exits in 21 years: the scorecard is the
evidence). Caveat: 2008 is known to everyone (hindsight-prone); the result does not change the grid or the selection.
Outputs: store04_experiments/exp07_warning_lights_exit/step01_exploration_data/. Trials: 3 (stage "exploration").
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp07_warning_lights_exit import config as exp_config
# IMPORT DATA LOADING, EXECUTION AND DAILY FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT EXIT AND RE-ENTRY SIMULATION FUNCTIONS
from so.core.reentry_simulation import get_buy_and_hold_result_dict
# IMPORT THE CONTINUOUS REPLAY FUNCTIONS
from so.core.continuous_replay import simulate_continuous_replay_dict, get_episode_scorecard_pdf, get_replay_summary_dict
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp07_warning_lights_exit.rules import get_rule_candidate_list, get_rule_builder_func, get_light_pdf
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [3]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_exploration_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "exploration")

Experiment:	exp07_warning_lights_exit	Protocol:	1.0	Config Hash:	2965fad3e3
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp07_warning_lights_exit/step01_exploration_data/
ℹ️ No earlier 'exploration' entry of exp07_warning_lights_exit in the trial log (config hash 2965fad3e3).


In [4]:
# CALL FUNCTION TO READ THE MINUTE BARS UP TO THE EXPLORATION DATA CUTOFF (NOTHING FROM THE WALK-FORWARD PERIOD IS READ)
complete_ohlcv_pdf = get_complete_ohlcv_pdf(cutoff_date_str_in=config.EXPLORATION_DATA_CUTOFF_DATE_STR)
# CALL FUNCTION TO CONVERT THE DATA TO ARRAYS AND BUILD THE DAILY TABLE
ohlcv_array_dict = get_ohlcv_array_dict(complete_ohlcv_pdf)
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
# DEFINE THE EXPLORATION WINDOW
exploration_start_str, exploration_end_str = config.EXPLORATION_START_DATE_STR, config.EXPLORATION_END_DATE_STR
# DEFINE ONE PERIOD PER CALENDAR YEAR (FOR THE YEARS-WON COUNT)
session_date_arr = np.array(ohlcv_array_dict["session_date_list"])
window_date_arr = session_date_arr[(session_date_arr >= pd.Timestamp(exploration_start_str).date()) & (session_date_arr <= pd.Timestamp(exploration_end_str).date())]
year_period_pdf = pd.DataFrame([{"period_id": year, "period_start": min(d for d in window_date_arr if d.year == year), "period_end": max(d for d in window_date_arr if d.year == year)}
                                for year in sorted({d.year for d in window_date_arr})])
# DISPLAY INFORMATION
print(f"Data:\t{daily_pdf['date'].min()} -> {daily_pdf['date'].max()}\t({len(daily_pdf):,} sessions)\tReplay:\t{exploration_start_str} -> {exploration_end_str}\t({len(window_date_arr):,} sessions, {len(year_period_pdf)} years)")
# COUNT THE LIGHTS OF THE WINDOW'S DECISIONS
light_pdf = get_light_pdf(daily_pdf)
window_light_pdf = light_pdf[(light_pdf["date"] >= pd.Timestamp(exploration_start_str).date()) & (light_pdf["date"] <= pd.Timestamp(exploration_end_str).date())]
print("Share of decisions with each light on:", window_light_pdf[list(exp_config.LIGHT_DICT)].mean().round(3).to_dict())
print("Share of decisions with at least k lights on:", {k: round(float((window_light_pdf["light_count"] >= k).mean()), 3) for k in range(1, 6)})

Data:	2005-01-03 -> 2015-03-18	(2,568 sessions)	Replay:	2005-01-03 -> 2014-12-31	(2,516 sessions, 10 years)
Share of decisions with each light on: {'below_ma200': 0.244, 'return_250d_negative': 0.19, 'volatility_rising': 0.187, 'ath_drawdown_10pct': 0.476, 'high60_stale': 0.4}
Share of decisions with at least k lights on: {1: 0.671, 2: 0.409, 3: 0.249, 4: 0.141, 5: 0.027}


In [5]:
"""
The 3 candidates, each one continuous path (start invested, no forced buy-back), against buy-and-hold.
"""
# SIMULATE BUY AND HOLD
buy_hold_dict = get_buy_and_hold_result_dict(ohlcv_array_dict, exploration_start_str, exploration_end_str)
print(f"Buy & hold:\ttotal return {buy_hold_dict['metric_dict']['total_return']:.2%} | Sharpe {buy_hold_dict['metric_dict']['sharpe_ratio']:.2f} | max drawdown {buy_hold_dict['metric_dict']['max_drawdown']:.2%}")
# DEFINE THE RULE BUILDER
rule_builder_func = get_rule_builder_func(daily_pdf)
# LISTS TO HOLD THE RESULTS
row_dict_list, scorecard_pdf_list = [], []
# ITERATE OVER THE CANDIDATES
for candidate_idx, candidate_dict in enumerate(get_rule_candidate_list()):
    # BUILD AND REPLAY THE CANDIDATE
    exit_signal_arr, reentry_func, reason_str = rule_builder_func(candidate_dict)
    simulation_dict = simulate_continuous_replay_dict(ohlcv_array_dict, exploration_start_str, exploration_end_str, exp_config.UNTOUCHED_START_DATE_STR,
                                                      volatility_arr_in=np.full(len(daily_pdf), np.nan), stop_k_in=None, reentry_func_in=reentry_func,
                                                      reentry_reason_str_in=reason_str, exit_signal_arr_in=exit_signal_arr, max_cash_sessions_in=exp_config.MAX_CASH_SESSIONS)
    # SUMMARIZE THE PATH (YEARS AS PERIODS)
    summary_dict = get_replay_summary_dict(simulation_dict["daily_equity_pdf"], buy_hold_dict["daily_equity_pdf"], year_period_pdf, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT)
    scorecard_pdf = get_episode_scorecard_pdf(simulation_dict, ohlcv_array_dict)
    scorecard_pdf_list.append(scorecard_pdf.assign(candidate_idx=candidate_idx, **candidate_dict))
    # STORE THE ROW
    row_dict_list.append({"candidate_idx": candidate_idx, **candidate_dict, "total_return": summary_dict["strategy_total_return"], "buy_hold_total_return": summary_dict["buy_hold_total_return"],
                          "excess_total_return": summary_dict["excess_total_return"], "annualized_return": summary_dict["strategy_annualized_return"],
                          "sharpe_ratio": summary_dict["strategy_sharpe_ratio"], "max_drawdown": summary_dict["strategy_max_drawdown"], "in_market_pct": summary_dict["in_market_pct"],
                          "exit_count": simulation_dict["metric_dict"]["signal_exit_count"], "mean_cash_sessions": simulation_dict["metric_dict"]["mean_cash_sessions"],
                          "bought_back_lower_pct": simulation_dict["metric_dict"]["positive_episode_pct"],
                          "mean_log_share_gain": float(scorecard_pdf["log_share_gain"].mean()) if len(scorecard_pdf) else np.nan,
                          "years_won": summary_dict["period_windows_beating_buy_hold"], "year_count": summary_dict["period_window_count"],
                          "annualized_log_excess": summary_dict["log_excess_annualized_log_excess"], "log_excess_ci_low": summary_dict["log_excess_ci_low"], "log_excess_ci_high": summary_dict["log_excess_ci_high"]})
# CONVERT THE RESULTS
exploration_pdf = pd.DataFrame(row_dict_list)
exploration_scorecard_pdf = pd.concat(scorecard_pdf_list, ignore_index=True)
# PREVIEW DATAFRAME
exploration_pdf.round(4)

Buy & hold:	total return 69.02% | Sharpe 0.36 | max drawdown -56.44%


,candidate_idx,light_k,total_return,buy_hold_total_return,excess_total_return,annualized_return,sharpe_ratio,max_drawdown,in_market_pct,exit_count,mean_cash_sessions,bought_back_lower_pct,mean_log_share_gain,years_won,year_count,annualized_log_excess,log_excess_ci_low,log_excess_ci_high
0,0,3,0.4271,0.6902,-0.2631,0.0363,0.4016,-0.2017,0.6588,14,61.2857,0.1429,-0.0120,2,10,-0.0169,-0.1109,0.1041
1,1,4,0.7456,0.6902,0.0555,0.0574,0.5145,-0.2624,0.8262,11,39.7273,0.0909,0.0030,3,10,0.0032,-0.0752,0.1033
2,2,5,1.3641,0.6902,0.6739,0.0900,0.6626,-0.2193,0.9070,2,117.0000,0.5000,0.1679,6,10,0.0336,-0.0256,0.1270


In [6]:
"""
Episode scorecard of every candidate (k = 3, 4, 5).
S / R > 1 = bought back lower than sold; buy_hold_return_out = what holding earned while out.
"""
# DISPLAY THE THREE SCORECARDS
scorecard_col_list = ["exit_date", "exit_fill_price", "reentry_date", "reentry_fill_price", "s_over_r", "sessions_out", "buy_hold_return_out", "max_decline_out_pct", "cost_paid_pct", "open_at_end"]
for light_k in exp_config.LIGHT_K_LIST:
    candidate_scorecard_pdf = exploration_scorecard_pdf[exploration_scorecard_pdf["light_k"] == light_k]
    print(f"\nk = {light_k}:\t{len(candidate_scorecard_pdf)} episodes, {int((candidate_scorecard_pdf['s_over_r'] > 1).sum())} bought back lower, "
          f"product of S/R {np.prod(candidate_scorecard_pdf['s_over_r']):.4f}")
    display(candidate_scorecard_pdf[scorecard_col_list].round(4))


k = 3:	14 episodes, 2 bought back lower, product of S/R 0.8450


,exit_date,exit_fill_price,reentry_date,reentry_fill_price,s_over_r,sessions_out,buy_hold_return_out,max_decline_out_pct,cost_paid_pct,open_at_end
0,2005-10-24,119.97,2005-11-17,124.53,0.9634,18,0.0378,-0.0170,0.0002,False
1,2006-06-07,126.01,2006-06-29,127.22,0.9905,16,0.0094,-0.0292,0.0002,False
2,2007-08-15,140.74,2007-09-04,149.31,0.9426,13,0.0607,-0.0266,0.0002,False
3,2007-11-14,147.16,2007-12-24,149.42,0.9849,27,0.0152,-0.0434,0.0002,False
4,2008-01-04,140.98,2009-10-05,104.03,1.3552,441,-0.2622,-0.5241,0.0003,False
5,2010-02-16,109.72,2010-03-10,115.01,0.9540,16,0.0480,-0.0072,0.0003,False
6,2010-05-20,107.65,2010-09-14,112.75,0.9548,80,0.0472,-0.0607,0.0003,False
7,2011-03-22,129.31,2011-04-26,134.81,0.9592,24,0.0424,-0.0077,0.0002,False
8,2011-06-15,127.06,2011-10-27,128.47,0.9890,94,0.0109,-0.1546,0.0002,False
9,2011-11-01,121.88,2011-11-08,127.89,0.9530,5,0.0491,-0.0002,0.0002,False



k = 4:	11 episodes, 1 bought back lower, product of S/R 1.0337


,exit_date,exit_fill_price,reentry_date,reentry_fill_price,s_over_r,sessions_out,buy_hold_return_out,max_decline_out_pct,cost_paid_pct,open_at_end
14,2007-11-26,140.94,2007-12-05,148.89,0.9466,7,0.0563,-0.0012,0.0002,False
15,2008-01-08,138.77,2008-05-06,141.85,0.9783,82,0.0220,-0.0921,0.0002,False
16,2008-06-18,134.24,2009-06-01,94.75,1.4168,239,-0.2943,-0.5002,0.0003,False
17,2010-05-25,107.76,2010-06-21,111.30,0.9682,18,0.0327,-0.0290,0.0003,False
18,2011-08-05,120.30,2011-10-27,128.47,0.9364,58,0.0677,-0.1071,0.0002,False
19,2011-11-28,119.60,2011-12-07,126.65,0.9443,7,0.0588,-0.0003,0.0002,False
20,2011-12-08,123.80,2011-12-23,126.37,0.9797,11,0.0206,-0.0305,0.0002,False
21,2011-12-28,124.75,2012-01-04,127.69,0.9770,4,0.0234,-0.0001,0.0002,False
22,2012-06-01,128.29,2012-06-05,129.07,0.9940,2,0.0059,-0.0090,0.0002,False
23,2012-06-28,132.87,2012-07-02,136.53,0.9732,2,0.0274,-0.0008,0.0002,False



k = 5:	2 episodes, 1 bought back lower, product of S/R 1.3991


,exit_date,exit_fill_price,reentry_date,reentry_fill_price,s_over_r,sessions_out,buy_hold_return_out,max_decline_out_pct,cost_paid_pct,open_at_end
25,2008-03-20,132.80,2008-04-24,138.88,0.9562,24,0.0456,-0.0132,0.0002,False
26,2008-06-27,127.69,2009-04-29,87.27,1.4632,210,-0.3167,-0.4745,0.0003,False


In [7]:
# SAVE THE RESULTS
write_csv_file_to_path(exploration_pdf, f"{output_path_str}exploration_candidate_data.csv", "W")
write_csv_file_to_path(exploration_scorecard_pdf, f"{output_path_str}exploration_scorecard_data.csv", "W")

ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp07_warning_lights_exit/step01_exploration_data/exploration_candidate_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp07_warning_lights_exit/step01_exploration_data/exploration_candidate_data.csv' Has Been Saved!
ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp07_warning_lights_exit/step01_exploration_data/exploration_scorecard_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp07_warning_lights_exit/step01_exploration_data/exploration_scorecard_data.csv' Has Been Saved!


In [8]:
# LOG THE 3 CANDIDATES AS EXPLORATION TRIALS (ONE WRITE)
exploration_trial_pdf = log_trial_pdf(exp_config, "exploration", exploration_pdf.assign(window=f"{exploration_start_str}->{exploration_end_str}"), ["light_k", "window"],
                                      [col for col in exploration_pdf.columns if col not in ["candidate_idx", "light_k"]], False,
                                      note_str_in="exp07 step01 exploration (2005-2014 continuous replay, context only)")
print(f"Trials logged:\t{len(exploration_trial_pdf)}")

Trials logged:	3


In [9]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp01_minute_entry,signal_check,2,2,0,2026-10-05T15:17:38,2026-10-06T12:46:46
1,exp01_minute_entry,summary,1,1,0,2026-10-05T16:16:30,2026-10-05T16:16:30
2,exp01_minute_entry,validation,540,1,0,2026-10-05T15:36:53,2026-10-05T16:16:18
3,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
4,exp02_stop_reentry,signal_check,1,1,0,2026-10-05T14:48:41,2026-10-05T14:48:41
5,exp02_stop_reentry,summary,1,1,0,2026-10-05T14:50:29,2026-10-05T14:50:29
6,exp02_stop_reentry,validation,792,1,0,2026-10-05T14:49:18,2026-10-05T14:49:48
7,exp03_ath_exit,exploration,15,1,0,2026-10-05T14:55:04,2026-10-05T14:55:04
8,exp03_ath_exit,summary,1,1,0,2026-10-05T14:56:49,2026-10-05T14:56:49
9,exp03_ath_exit,validation,660,1,0,2026-10-05T14:55:55,2026-10-05T14:56:12
